# Inpainting Reviewer
Review first/last frames (original + inpainted) and flag failures. This notebook leaves the HTML viewer untouched.


In [ ]:
import json, pickle, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import widgets, Layout, HBox, VBox
from IPython.display import display, clear_output

In [ ]:
# --- Configuration ---
# Path to PKL containing a dict with key 'trajectories'
# dataset_path = 'data/bridge_fractal_n300_with_inpainted.pkl'
dataset_path = 'data/bridge_fractal_train_n200_inpainted.pkl'
# Optional JSON file of failed ids. Supported formats:
# 1) [1,2,3] (applies to all datasets; ambiguous if ids collide)
# 2) [{"dataset": "bridge", "trajectory_id": 1}, ...]
# 3) {"bridge": [1,2], "fractal": [3,4]}
failed_ids_path = None
# Outputs
output_json = 'flags_inpaintings_train.json'
output_csv = 'flags_inpaintings_train.csv'
# Persist dataset with embedded failed_ids/failed_records (set same as dataset_path to overwrite)
# output_dataset_path = 'data/bridge_fractal_n300_with_inpainted_cleaned.pkl'
output_dataset_path = 'data/bridge_fractal_train_n200_inpainted_cleaned.pkl'
# If True, only review trajectories listed in failed ids
only_failed_candidates = False
# Index to start from
start_index = 0

In [ ]:
def load_failed_ids(path):
    if path is None:
        return set(), None
    p = Path(path)
    if not p.exists():
        print(f"Failed ids file not found: {p}")
        return set(), None
    with open(p, 'r', encoding='utf-8') as f:
        data = json.load(f)
    pairs = set()
    note = None
    if isinstance(data, list):
        if all(isinstance(x, int) for x in data):
            note = 'List of ints provided; applies to all datasets (ambiguous if ids collide).'
            pairs = {(None, int(x)) for x in data}
        elif all(isinstance(x, dict) and 'trajectory_id' in x for x in data):
            for x in data:
                pairs.add((x.get('dataset'), int(x['trajectory_id'])))
        else:
            note = 'Unrecognized list format; ignoring.'
    elif isinstance(data, dict):
        for ds, ids in data.items():
            for i in ids:
                pairs.add((ds, int(i)))
    else:
        note = 'Unrecognized failed-ids format; ignoring.'
    return pairs, note


def load_dataset(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Missing dataset file: {p}")
    with open(p, 'rb') as f:
        data = pickle.load(f)
    trajectories = data['trajectories'] if isinstance(data, dict) and 'trajectories' in data else data
    return data, trajectories


def make_key(traj, fallback_id):
    ds = traj.get('dataset', 'unknown')
    tid = traj.get('trajectory_id', fallback_id)
    return f"{ds}:{tid}", ds, tid


In [ ]:
base_data, trajectories = load_dataset(dataset_path)
failed_pairs, failed_note = load_failed_ids(failed_ids_path)
print(f"Loaded {len(trajectories)} trajectories")
if failed_note:
    print(failed_note)


def select_indices(trajs, failed_pairs, only_failed):
    idxs = []
    for i, t in enumerate(trajs):
        key, ds, tid = make_key(t, i)
        if only_failed:
            if (ds, tid) in failed_pairs or (None, tid) in failed_pairs:
                idxs.append(i)
        else:
            idxs.append(i)
    return idxs


candidate_indices = select_indices(trajectories, failed_pairs, only_failed_candidates)
if not candidate_indices:
    print('No trajectories selected; check failed_ids_path or disable only_failed_candidates.')
current_pos = min(start_index, max(len(candidate_indices) - 1, 0))

flags = {}  # key -> dict(dataset, trajectory_id, first_bad, last_bad, note)

print(f"Reviewing {len(candidate_indices)} trajectories")


In [ ]:
from ipywidgets import widgets, Layout, HBox, VBox, Output
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

def get_image(traj, key, fallback_shape=(256, 256, 3)):
    img = traj.get(key)
    if img is None:
        return np.ones(fallback_shape, dtype=np.uint8) * 255
    arr = np.array(img)
    if arr.dtype != np.uint8:
        arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
    return arr


def render(idx, info_widget, note_widget, status_widget, plot_output):
    """Render trajectory at index. Requires live widget references."""
    traj_idx = candidate_indices[idx]
    traj = trajectories[traj_idx]
    key, ds, tid = make_key(traj, traj_idx)
    flag = flags.get(key, {'first_bad': False, 'last_bad': False, 'note': ''})

    # Clear only the plot output area
    with plot_output:
        clear_output(wait=True)
        
        # Use GridSpec for custom column widths (gap is 50% width)
        fig = plt.figure(figsize=(18, 4))
        gs = fig.add_gridspec(1, 5, width_ratios=[1, 1, 0.5, 1, 1])
        
        ax0 = fig.add_subplot(gs[0])
        ax1 = fig.add_subplot(gs[1])
        ax2 = fig.add_subplot(gs[2])
        ax3 = fig.add_subplot(gs[3])
        ax4 = fig.add_subplot(gs[4])
        
        ax0.imshow(get_image(traj, 'first_image'))
        ax0.set_title('first original')
        ax1.imshow(get_image(traj, 'first_image_inpainted'))
        ax1.set_title('first inpainted')
        ax2.axis('off')  # Gap
        ax3.imshow(get_image(traj, 'last_image'))
        ax3.set_title('last original')
        ax4.imshow(get_image(traj, 'last_image_inpainted'))
        ax4.set_title('last inpainted')
        
        for ax in [ax0, ax1, ax2, ax3, ax4]:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    # Display inpainting prompt if available, otherwise show target object
    inpaint_info = traj.get('inpainting_prompt', traj.get('language_target_object', '(not stored)'))
    
    # Highlight Language field with yellow background and larger font
    info_html = f"""<b>Dataset:</b> {ds} &nbsp; <b>Trajectory ID:</b> {tid}<br>
<div style="background-color: #ffeb3b; padding: 8px; border-radius: 4px; margin: 8px 0;">
    <span style="font-size: 16px; font-weight: bold; color: #000;">Language:</span>
    <span style="font-size: 16px;">{traj.get('language', 'n/a')}</span>
</div>
<b>Target removed:</b> {inpaint_info}<br>
<b>Flag status:</b> first_bad={flag['first_bad']}, last_bad={flag['last_bad']}"""
    info_widget.value = info_html
    note_widget.value = flag.get('note', '')
    status_widget.value = f"{idx + 1}/{len(candidate_indices)} | key={key}"


def save_flags():
    payload = list(flags.values())
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)
    with open(output_csv, 'w', encoding='utf-8') as f:
        f.write('dataset,trajectory_id,first_bad,last_bad,note\n')
        for r in payload:
            note = (r.get('note') or '').replace('\\', ' ').replace('\n', ' ')
            f.write(f"{r['dataset']},{r['trajectory_id']},{int(r['first_bad'])},{int(r['last_bad'])},\"{note}\"\n")
    print(f"Saved {len(payload)} flags to {output_json} and {output_csv}")


def save_dataset_with_flags():
    # SAFETY CHECK: prevent overwriting original dataset
    if Path(output_dataset_path) == Path(dataset_path):
        raise ValueError(f"SAFETY: Output path equals input path! Refusing to overwrite original dataset.\n"
                        f"  Input:  {dataset_path}\n"
                        f"  Output: {output_dataset_path}")
    
    # keep structure identical to input; add failed_ids/failed_records
    failed_records = []
    seen = set()
    for r in flags.values():
        if r.get('first_bad') or r.get('last_bad'):
            key = f"{r['dataset']}:{r['trajectory_id']}"
            if key in seen:
                continue
            seen.add(key)
            failed_records.append({
                'dataset': r['dataset'],
                'trajectory_id': r['trajectory_id'],
                'first_bad': bool(r.get('first_bad', False)),
                'last_bad': bool(r.get('last_bad', False)),
                'note': r.get('note', ''),
            })

    failed_ids = [{'dataset': fr['dataset'], 'trajectory_id': fr['trajectory_id']} for fr in failed_records]

    if isinstance(base_data, dict):
        new_data = dict(base_data)
        new_data['failed_ids'] = failed_ids
        new_data['failed_records'] = failed_records
    else:
        # Fallback: wrap list into dict to avoid losing info
        new_data = {'trajectories': base_data, 'failed_ids': failed_ids, 'failed_records': failed_records}

    with open(output_dataset_path, 'wb') as f:
        pickle.dump(new_data, f)
    print(f"Saved dataset with flags to {output_dataset_path} ({len(failed_records)} failed entries)")

In [ ]:
# Widgets
prev_btn = widgets.Button(description='Prev', layout=Layout(width='80px'))
next_btn = widgets.Button(description='Next', layout=Layout(width='80px'))
fail_first_btn = widgets.Button(description='Fail First', button_style='danger')
fail_last_btn = widgets.Button(description='Fail Last', button_style='danger')
fail_both_btn = widgets.Button(description='Fail Both', button_style='danger')
pass_btn = widgets.Button(description='Mark Pass/Skip', button_style='success')
save_btn = widgets.Button(description='Save Flags', button_style='info')
save_dataset_btn = widgets.Button(description='Save Dataset+Flags', button_style='info')
info = widgets.HTML()
status = widgets.Label()
note_box = widgets.Textarea(placeholder='Optional note for this trajectory')
plot_output = widgets.Output()

nav_box = HBox([prev_btn, next_btn, status])
flag_box = HBox([fail_first_btn, fail_last_btn, fail_both_btn, pass_btn, save_btn, save_dataset_btn])
ui = VBox([nav_box, info, note_box, flag_box, plot_output])

In [ ]:
def update_flag(first_bad=None, last_bad=None):
    traj_idx = candidate_indices[current_pos]
    traj = trajectories[traj_idx]
    key, ds, tid = make_key(traj, traj_idx)
    record = flags.get(key, {'dataset': ds, 'trajectory_id': tid, 'first_bad': False, 'last_bad': False, 'note': ''})
    if first_bad is not None:
        record['first_bad'] = first_bad
    if last_bad is not None:
        record['last_bad'] = last_bad
    record['note'] = note_box.value
    flags[key] = record
    render(current_pos, info, note_box, status, plot_output)


def on_prev(_):
    global current_pos
    if current_pos > 0:
        current_pos -= 1
        render(current_pos, info, note_box, status, plot_output)


def on_next(_):
    global current_pos
    if current_pos < len(candidate_indices) - 1:
        current_pos += 1
        render(current_pos, info, note_box, status, plot_output)


def on_fail_first(_):
    update_flag(first_bad=True)


def on_fail_last(_):
    update_flag(last_bad=True)


def on_fail_both(_):
    update_flag(first_bad=True, last_bad=True)


def on_pass(_):
    update_flag(first_bad=False, last_bad=False)


def on_save(_):
    save_flags()


def on_save_dataset(_):
    save_flags()
    save_dataset_with_flags()


prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
fail_first_btn.on_click(on_fail_first)
fail_last_btn.on_click(on_fail_last)
fail_both_btn.on_click(on_fail_both)
pass_btn.on_click(on_pass)
save_btn.on_click(on_save)
save_dataset_btn.on_click(on_save_dataset)

In [ ]:
display(ui)
if candidate_indices:
    render(current_pos, info, note_box, status, plot_output)
else:
    print('Nothing to review.')

In [ ]:
# train:
# bridge:10 has nonsensical prompt, bridge:41 about robotic arm, bridge:43 same, 

In [ ]:
# rules for future inpainting:
# if talking about moving a cup or can and there are multiple in image that look similar, remove all
# - make sure robot arm did not change
# drawers never disappear --> more explicit prompt

In [ ]:
# bridge:29 has not inpainted images?, bridge:40 too
# bridge:56 is anomaly, command is not about doing something but asking "did something happen", bridge:57 too!?
# bridge:78 does not match the image?
# bridge:114 and 115 have no change at all in them
# bridge:119 has language prompt fghgfjf, is that real?
# bridge 139: first and last inversed
# fractal:10 has no inpaintings, fractal:27 too, fractal:51 has last picture no inpainting
# fractal:131 has no inpaintings, 

In [ ]:
import pickle

# Load the cleaned file
with open('data/bridge_fractal_n300_with_inpainted_cleaned.pkl', 'rb') as f:
    cleaned = pickle.load(f)

print(f"Total trajectories: {len(cleaned['trajectories'])}")
print(f"Failed IDs: {len(cleaned.get('failed_ids', []))}")
print(f"Failed Records: {len(cleaned.get('failed_records', []))}")

# Show first few failed records
if cleaned.get('failed_records'):
    print("\nFirst 5 failed records:")
    for r in cleaned['failed_records'][:5]:
        print(f"  {r['dataset']}:{r['trajectory_id']} - first_bad={r['first_bad']}, last_bad={r['last_bad']}")